<a href="https://colab.research.google.com/github/sanadv/Classical_Machine_Learning/blob/main/Classical_Machine_Leaning_Chapter_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from PIL import Image
# Load the image and ensure it's in RGB format
img = Image.open('Lenna_(test_image).png').convert('RGB')
img = img.resize((256, 256))  # Resize for simplicity if needed

# Convert the image to a NumPy array and normalize it
img_array = np.array(img, dtype='float32') / 255.0  # Normalize to [0, 1]
img_processed = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Define a Conv2D layer to extract feature maps
conv_layer = tf.keras.layers.Conv2D(filters=4, kernel_size=(3, 3), strides=(1, 1), padding='same', activation=None)
feature_maps = conv_layer(img_processed)

# Extract the feature maps
feature_maps = feature_maps[0].numpy()

# Plot the original image
plt.figure(figsize=(12, 6))
plt.subplot(1, 5, 1)
plt.imshow(img)
plt.title("Original Image")
plt.axis("off")

# Plot the resulting feature maps
for i in range(4):
    plt.subplot(1, 5, i + 2)
    plt.imshow(feature_maps[:, :, i], cmap='gray')
    plt.title(f"Feature Map {i + 1}")
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
#Cubixel Kernel
class CubixelLayer(tf.keras.layers.Layer):
    def __init__(self, num_kernels, kernel_size=(3, 3), volume_threshold=1.0, **kwargs):
        super(CubixelLayer, self).__init__(**kwargs)
        self.num_kernels = num_kernels
        self.kernel_size = kernel_size
        self.volume_threshold = volume_threshold
        self.edge_color = np.array([255, 0, 0], dtype=np.float32)  # Red color for edges
        self.kernels = self.add_weight(
            name='kernels',
            shape=(num_kernels, kernel_size[0], kernel_size[1], 1, 1),
            initializer='random_normal',
            trainable=True
        )

    def call(self, inputs):
        # Calculate cubixel properties
        volumes, aspect_ratios = self.calculate_cubixel_properties(inputs)

        kernel_outputs = []
        for i in range(self.num_kernels):
            kernel = self.kernels[i]
            altered_volumes, altered_aspect_ratios = self.apply_magnetic_kernel((volumes, aspect_ratios), kernel)
            local_threshold = self.calculate_local_threshold(altered_volumes)

            # Detect edges at the current scale
            edges = tf.cast(altered_volumes, tf.float32) > tf.cast(local_threshold, tf.float32)
            edges |= tf.cast(altered_aspect_ratios, tf.float32) > tf.cast(local_threshold, tf.float32)

            # Apply edge color and store each kernel's output separately
            edge_colored_image = tf.cast(edges, tf.float32)[..., tf.newaxis] * self.edge_color
            kernel_outputs.append(edge_colored_image)

        # Stack all kernel outputs along a new axis (axis=0)
        # This will give you a tensor of shape [num_kernels, height, width, channels]
        multi_scale_edges = tf.stack(kernel_outputs, axis=0)

        return multi_scale_edges

    @staticmethod
    def calculate_cubixel_properties(image):
        # Normalize the image to a range of 0-1
        image_normalized = image / 255.0

        # Calculate volumes as the product of the RGB values (considered as dimensions)
        # Assuming image shape: [batch, height, width, channels]
        # Volumes are calculated for each pixel
        volumes = image_normalized[..., 0] * image_normalized[..., 1] * image_normalized[..., 2]

        # Calculate aspect ratios
        # Example: You could define aspect ratios based on the relative proportions of RGB values
        # Adjust the calculation based on your specific definition of aspect ratio
        aspect_ratios = image_normalized[..., 0] / image_normalized[..., 1]  # Placeholder, replace with your aspect ratio calculation

        return volumes, aspect_ratios

    def apply_magnetic_kernel(self, cubixel_properties, kernel):
        kernel_tf = tf.reshape(kernel, [self.kernel_size[0], self.kernel_size[1], 1, 1])
        # Ensure that the input to the convolution is 4D by adding a channel dimension.
        altered_volume = tf.nn.conv2d(cubixel_properties[0][..., tf.newaxis], kernel_tf, strides=[1, 1, 1, 1], padding="SAME")
        altered_aspect_ratio = tf.nn.conv2d(cubixel_properties[1][..., tf.newaxis], kernel_tf, strides=[1, 1, 1, 1], padding="SAME")
        # Remove the last dimension to maintain the original shape.
        return altered_volume[..., 0], altered_aspect_ratio[..., 0]
    def calculate_local_threshold(self, image, neighborhood_size=9):
        pad_size = neighborhood_size // 2
        image = tf.expand_dims(image, -1) if len(image.shape) == 3 else image

        padded_images = tf.pad(
            image,
            paddings=[[0, 0], [pad_size, pad_size], [pad_size, pad_size], [0, 0]],
            mode='REFLECT'
        )

        patches = tf.image.extract_patches(
            images=padded_images,
            sizes=[1, neighborhood_size, neighborhood_size, 1],
            strides=[1, 1, 1, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )

        local_thresholds = tf.reduce_mean(patches, axis=-1, keepdims=True)

        # Ensure the output has the same shape as the input, minus the channel dimension
        local_thresholds = tf.squeeze(local_thresholds, -1)

        return local_thresholds


from PIL import Image

# Load the image and ensure it's in RGB format
img = Image.open('Lenna_(test_image).png').convert('RGB')
img = img.resize((256, 256))  # Resize for simplicity if needed

# Convert the image to a NumPy array and normalize it
img_array = np.array(img, dtype='float32') / 255.0  # Normalize to [0, 1]
img_processed = np.expand_dims(img_array, axis=0)  # Add batch dimension

# Define a Conv2D layer to extract feature maps

# Normalize the data
# Define the volume threshold
volume_threshold = 1.0
number_of_kernals = 4
# Initialize the custom layer with the defined volume threshold
cubixel_layer = CubixelLayer(number_of_kernals,
                             kernel_size=(3, 3),
                             volume_threshold=volume_threshold)

# Normalize the data

# Apply the custom layer to the first 10 images
processed_images = cubixel_layer(img_processed)

# Rest of your plotting code...
# Assuming processed_images is a tensor of shape [num_kernels, num_images, height, width, channels]

num_kernels = processed_images.shape[0]  # Number of kernels in the CubixelLayer
num_images = processed_images.shape[1]   # Number of images processed

# Set up the plotting
# Calculate the total number of subplots needed (original + number of kernels for each image)
plt.figure(figsize=(20, num_images * 2))

for i in range(num_images):
    # Original image subplot
    plt.subplot(num_images, num_kernels + 1, i * (num_kernels + 1) + 1)
    plt.imshow(img)
    plt.title("Original")
    plt.axis("off")

    # Processed images subplots
    for j in range(num_kernels):
        # Select the processed image for the current kernel
        processed_image = processed_images[j, i, :, :, 0]  # Assuming the last dimension is for grayscale

        plt.subplot(num_images, num_kernels + 1, i * (num_kernels + 1) + 2 + j)
        plt.imshow(processed_image, cmap='gray')  # Use a grayscale colormap
        plt.title(f"Kernel {j+1}")
        plt.axis("off")

# Adjust layout to prevent overlapping titles and axes
plt.tight_layout()
# Display the plot
plt.show()
